# Datamine PCA Process Example

This notebook demonstrates how to configure and run the **`pca`** process wrapper in `dmstudio`.

## Process Description

## PCA

Principal Components Analysis groups fields together into components on the basis of the correlation (R) or variance/covariance (C) matrix.

Each component is determined to maximize the total variance. Note, this is different to Factor Analysis which maximizes the inter-correlation between components. Multiple options are available by selecting the type of matrix, type of principal component loading and type of scores to calculate for each sample.

The number of groups or components required can be set directly by setting the number of eigen values (@**NUMEIGN**). If the default value of zero is entered then the number of components selected will be as follows:

* R-matrix, number of eigen values exceeding @**EIGENMIN** , default is 1.0.

* C-matrix, number of eigen values whose cumulative percentage of variance exceeds @**MAXVAR** , default is 95 percent.

* Number of fields selected for analysis.

### File Handling

The input file (&**IN**) must have a separate sample identifier field (* **SAMPID**) which has to be declared on input. There is the optional facility (&**SCORES**) to output the calculated principal component scores for farther processing.

If the user wishes to plot maps of the output scores then the **SCORES** file can be joined to the original input file using the **[JOIN](<join.md>)** process and defining **SAMPID** as the keyfield.

### Process Limits

There is currently a restriction of 45 variables. There is no limit on the number of samples.

On the basis of **MATXTYPE, LOADEIGN** and **SCNORM** there are eight options available for PCA analysis, the following five are recommended:

## MATXTYPE |  LOADEIGN |  SCNORM

---|---|---

* **0** (R-Matrix  |  1: Eigen vector scores  |  1: Scores not normalized):

* **0** (R-Matrix  |  0: PC factor loadings  |  0: Normalized scores):

* **1** (C-Matrix  |  1: Eigen vector scores  |  1: Scores not normalized):

* **1** (C-Matrix  |  1: Eigen vector scores  |  0: Normalized scores):

* **1** (C-matrix  |  0: PC factor loadings  |  0: Normalized scores):

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **scores** (Undefined):
  Optional output file for principal component scores.
  Required=No

### Fields:

* **sampid** (Any : IN):
  Field containing sample identification
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **matxtype**:
  Matrix type to be used to calculate components. Options: (0): R matrix, standardised Z
  values of original data.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **eigenmin**:

* **Options** ((1): Eigenvalue required to select the number of components. Used in the):
  R-matrix only, ie. when MATXTYPE = 0. Used also when NUMEIGEN = 0.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **numeigen**:

* **Options** ((0): Maximum number of eigenvalues is set to the number of fields or to 10,):
  whichever is the lower. Note that if a non default value of MAXVARPC is used, NUMEIGEN
  must be 0.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **maxvarpc**:
  Specific to the selection of the variance/covariance matrix for analysis ie.
  MATXTYPE=1. The cumulative percentage of variation (95) required from the eigen values
  to select the number of eigen values for the analysis. Used when NUMEIGEN=0.
  Range=Undefined
  Values=Undefined
  Default=95
  Required=No

* **scnorm**:

* **Options** ((0): Normalised scores calculated.; 1: Scores are not normalised.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **loadeign**:

* **Options** ((0): Use factor loadings to calculate scores.; 1: Use eigenvalues to calculate):
  scores.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  > 0 Display scores on the screen (0). Note - Do not use for large files.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('pca')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute pca
print("Running pca...")
dm_cmd.pca(
    in_i='t_assays',  # required
    scores_o=['t_pca_out'],  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # matxtype_p=0,  # optional
    # eigenmin_p=1,  # optional
    # numeigen_p=0,  # optional
    # maxvarpc_p=95,  # optional
    # scnorm_p=0,  # optional
    # loadeign_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("pca execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_pca_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")